# Compile LFM2.5-230M FHE circuits for GPU (Akash deployment)

This notebook compiles the LFM2.5-230M FHE inference server artifacts for **GPU** using Zama Concrete-ML.
The output `server.zip` files are designed to be deployed inside the `ghcr.io/localchimera/fhe-inference` image on Akash.

**Note:** Kaggle's default Python 3.12 cannot install `concrete-ml==1.9.0` because `onnxoptimizer==0.3.13` has no Python 3.12 wheel. This notebook installs Python 3.11 and runs the compilation under Python 3.11.

## Steps
1. Turn on the GPU accelerator (Runtime → Change runtime type → GPU T4).
2. Run all cells.
3. Download the `artifacts.zip` file from `/kaggle/working/output`.
4. Place the unzipped `lfm2_fhe_phase1/server.zip` and `lfm2_fhe_phase2/server.zip` into your Akash Docker build.


In [ ]:
# Install Python 3.11 and GPU dependencies for FHE compilation
!apt-get update -qq
!apt-get install -y -qq software-properties-common
!add-apt-repository -y ppa:deadsnakes/ppa
!apt-get update -qq
!apt-get install -y -qq python3.11 python3.11-venv
!python3.11 -m ensurepip --upgrade
!python3.11 -m pip install -q --upgrade pip

# Install concrete-ml and the GPU concrete-python wheel
!python3.11 -m pip install -q concrete-ml==1.9.0 transformers safetensors huggingface-hub
!python3.11 -m pip uninstall -y -q concrete-python
!python3.11 -m pip install -q --trusted-host pypi.zama.ai --extra-index-url https://pypi.zama.ai/gpu concrete-python==2.10.0
!python3.11 -m pip install -q --index-url https://download.pytorch.org/whl/cu118 torch==2.3.1+cu118

!python3.11 -c "import concrete.compiler; print('GPU available:', concrete.compiler.check_gpu_available()); print('GPU enabled:', concrete.compiler.check_gpu_enabled())"


## Copy the compilation script
The compilation script is self-contained below so the notebook does not depend on the repository being cloned.

In [ ]:
import os
import textwrap

compile_script = r'''
import os
import time
import json
import gc
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn

MODEL_ID = "LiquidAI/LFM2.5-230M"
ARTIFACTS_DIR = Path("/kaggle/working/output/lfm2_fhe")
PHASE1_DIR = Path("/kaggle/working/output/lfm2_fhe_phase1")
PHASE2_DIR = Path("/kaggle/working/output/lfm2_fhe_phase2")

def load_model_weights():
    from huggingface_hub import hf_hub_download, list_repo_files
    from safetensors.torch import load_file

    print(f"Loading {MODEL_ID} weights directly...")
    config_path = hf_hub_download(MODEL_ID, "config.json")
    with open(config_path) as f:
        config = json.load(f)

    hidden_size = config["hidden_size"]
    layer_types = config.get("layer_types", [])
    print(f"  Hidden size: {hidden_size}")
    print(f"  Layer types: {layer_types}")

    files = list_repo_files(MODEL_ID)
    safetensors_files = [f for f in files if f.endswith(".safetensors")]

    all_weights = {}
    for sf in safetensors_files:
        path = hf_hub_download(MODEL_ID, sf)
        weights = load_file(path)
        all_weights.update(weights)
        print(f"  Loaded {sf}: {len(weights)} tensors")

    return all_weights, config, hidden_size, layer_types

def build_phase1(weights, config, layer_idx):
    hidden_size = config["hidden_size"]
    intermediate_size = config.get("intermediate_size", 2560)
    prefix = f"model.layers.{layer_idx}."

    q_w = weights[f"{prefix}self_attn.q_proj.weight"].float()
    k_w = weights[f"{prefix}self_attn.k_proj.weight"].float()
    v_w = weights[f"{prefix}self_attn.v_proj.weight"].float()
    w1_w = weights[f"{prefix}feed_forward.w1.weight"].float()
    w3_w = weights[f"{prefix}feed_forward.w3.weight"].float()

    merged = torch.cat([q_w, k_w, v_w, w1_w, w3_w], dim=0)
    module = nn.Linear(hidden_size, merged.shape[0], bias=False)
    module.weight.data = merged
    print(f"  Phase 1 merged: {sum(p.numel() for p in module.parameters())/1e6:.1f}M params, weight {merged.shape}")
    return module, hidden_size

def build_phase2(weights, config, layer_idx):
    hidden_size = config["hidden_size"]
    intermediate_size = config.get("intermediate_size", 2560)
    prefix = f"model.layers.{layer_idx}."

    o_w = weights[f"{prefix}self_attn.out_proj.weight"].float()
    w2_w = weights[f"{prefix}feed_forward.w2.weight"].float()

    input_size = 1024 + intermediate_size
    output_size = hidden_size * 2
    merged = torch.zeros(output_size, input_size)
    merged[:hidden_size, :1024] = o_w
    merged[hidden_size:, 1024:] = w2_w

    module = nn.Linear(input_size, output_size, bias=False)
    module.weight.data = merged
    print(f"  Phase 2 merged: {sum(p.numel() for p in module.parameters())/1e6:.1f}M params, weight {merged.shape}")
    return module, hidden_size

def compile_phase(module, phase_name, input_shape, n_bits, p_error, output_dir, device="cpu"):
    from concrete.ml.torch.compile import compile_torch_model

    print(f"\nCompiling {phase_name}...")
    print(f"  Input shape: {input_shape}")
    print(f"  n_bits: {n_bits}, p_error: {p_error}, device: {device}")

    # Keep everything on CPU; compile_torch_model handles device placement internally
    calib_input = torch.randn(*input_shape)

    import resource
    mem_before = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
    print(f"  RAM before: {mem_before:.0f} MB")

    start = time.time()
    fhe_circuit = compile_torch_model(
        module,
        calib_input,
        n_bits=n_bits,
        p_error=p_error,
        device=device,
    )
    compile_time = time.time() - start
    mem_after = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024

    print(f"  Compilation: {compile_time:.1f}s")
    print(f"  RAM peak: {mem_after:.0f} MB")

    from concrete.ml.deployment import FHEModelDev
    output_dir.mkdir(parents=True, exist_ok=True)
    for f in output_dir.iterdir():
        f.unlink()

    fhe_dev = FHEModelDev(output_dir, fhe_circuit)
    fhe_dev.save()
    print(f"  Saved to {output_dir}")

    return fhe_circuit, compile_time

def main():
    n_bits = 5
    p_error = 0.02
    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("=" * 60)
    print("LFM2.5-230M Optimized FHE Compilation for GPU/Akash")
    print(f"  n_bits: {n_bits}")
    print(f"  p_error: {p_error}")
    print(f"  device: {device}")
    print("=" * 60)

    weights, config, hidden_size, layer_types = load_model_weights()
    attn_layer_idx = next((i for i, lt in enumerate(layer_types) if lt == "full_attention"), 2)
    print(f"\nUsing layer {attn_layer_idx} (first full_attention)")

    intermediate_size = config.get("intermediate_size", 2560)
    phase1_output_size = 1024 + 512 + 512 + intermediate_size + intermediate_size
    phase2_input_size = 1024 + intermediate_size
    phase2_output_size = hidden_size + hidden_size

    results = {}

    # Phase 1
    print(f"\n{'─' * 40}")
    print("Phase 1: QKV + Gate/Up projections")
    print(f"  Input: (1, {hidden_size}) -> Output: (1, {phase1_output_size})")
    print(f"  5 Linear ops in single FHE call")
    print(f"{'─' * 40}")

    module1, _ = build_phase1(weights, config, attn_layer_idx)
    fhe_circuit1, compile_time1 = compile_phase(
        module1, "Phase 1",
        input_shape=(1, hidden_size),
        n_bits=n_bits, p_error=p_error,
        output_dir=PHASE1_DIR,
        device=device,
    )
    results["phase1"] = {"compile_time": compile_time1}

    del module1, fhe_circuit1
    gc.collect()

    # Phase 2
    print(f"\n{'─' * 40}")
    print("Phase 2: Output + Down projections")
    print(f"  Input: (1, {phase2_input_size}) -> Output: (1, {phase2_output_size})")
    print(f"  2 Linear ops in single FHE call")
    print(f"{'─' * 40}")

    module2, _ = build_phase2(weights, config, attn_layer_idx)
    del weights
    gc.collect()

    fhe_circuit2, compile_time2 = compile_phase(
        module2, "Phase 2",
        input_shape=(1, phase2_input_size),
        n_bits=n_bits, p_error=p_error,
        output_dir=PHASE2_DIR,
        device=device,
    )
    results["phase2"] = {"compile_time": compile_time2}

    # Metadata
    ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
    metadata = {
        "model_id": MODEL_ID,
        "model_type": "lfm2_two_phase_fhe",
        "layer_index": attn_layer_idx,
        "hidden_size": hidden_size,
        "intermediate_size": intermediate_size,
        "n_bits": n_bits,
        "p_error": p_error,
        "device": device,
        "phase1_dir": str(PHASE1_DIR),
        "phase2_dir": str(PHASE2_DIR),
        "results": results,
    }
    with open(ARTIFACTS_DIR / "metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)

    print("\n" + "=" * 60)
    print("GPU FHE Compilation Complete")
    print("=" * 60)
    for phase, data in results.items():
        print(f"  {phase}: compile={data['compile_time']:.1f}s")
    print(f"  Phase 1: {PHASE1_DIR}")
    print(f"  Phase 2: {PHASE2_DIR}")
    print(f"  Metadata: {ARTIFACTS_DIR / 'metadata.json'}")
    print("=" * 60)

if __name__ == "__main__":
    main()
'''

with open('/kaggle/working/compile_lfm2_gpu.py', 'w') as f:
    f.write(compile_script)
print("Wrote /kaggle/working/compile_lfm2_gpu.py")


In [ ]:
# Compile both phases using Python 3.11
!python3.11 /kaggle/working/compile_lfm2_gpu.py

In [ ]:
# Zip the artifacts so you can download them from Kaggle
import shutil
from pathlib import Path

output_zip = Path("/kaggle/working/artifacts.zip")
shutil.make_archive(
    base_name=str(output_zip.with_suffix("")),
    format="zip",
    root_dir="/kaggle/working/output",
)
print(f"Artifacts ready: {output_zip}")
print(f"Size: {output_zip.stat().st_size / 1024 / 1024:.1f} MB")

## Next steps for Akash

1. Download `artifacts.zip` from the Kaggle output panel.
2. Extract it locally.
3. Place the artifacts in your Akash build context:
   - `lfm2_fhe_phase1/server.zip` -> `infra/akash-fhe/server_files/phase1/server.zip`
   - `lfm2_fhe_phase2/server.zip` -> `infra/akash-fhe/server_files/phase2/server.zip`
4. Build and push the GPU Docker image (`ghcr.io/localchimera/fhe-inference:gpu`).
5. Deploy on Akash using the `fhe-rtx4090` or `fhe-a100` GPU tier.
